# 🦋 Biomorphes de Pickover — Exploration ultra-complète

Ce notebook est un **laboratoire visuel interactif** dédié aux **biomorphes**, une famille de
fractales découverte "par accident" par **Clifford Pickover** en 1986 chez IBM, en modifiant
légèrement l'algorithme classique des ensembles de Julia. Le résultat : des formes évoquant des
insectes, des microbes, des créatures cellulaires — d'où le nom.

## Principe mathématique

Comme pour un ensemble de Julia classique, on itère une fonction complexe :

$$
z_{n+1} = f(z_n) + c
$$

où $f$ est typiquement $z \mapsto z^{p}$ (avec $p$ la *puissance*), et $c$ une constante complexe fixée.

**La différence clé** introduite par Pickover porte sur le **critère de divergence** : au lieu de
tester le module $|z_n|$ comme dans un Julia set classique, on teste **indépendamment** la partie
réelle et la partie imaginaire :

$$
\text{le point diverge si } |\,\mathrm{Re}(z_n)\,| > B \quad \textbf{ou} \quad |\,\mathrm{Im}(z_n)\,| > B
$$

Ce simple changement — tester séparément deux axes au lieu du module global — brise la symétrie
radiale des ensembles de Julia et fait apparaître des **filaments, pattes, antennes** : c'est ce qui
donne aux biomorphes leur allure organique si caractéristique.

## Ce que propose ce notebook

1. Génération rapide (vectorisée NumPy + accélération **Numba**)
2. Rendu binaire (silhouette de l'"insecte", fidèle à l'algorithme original de Pickover)
3. Rendu en **temps d'échappement** (coloration façon fractale, plus riche visuellement)
4. **Exploration interactive** avec sliders — puissance, constante $c$, bailout, itérations, zoom, formule
5. Plusieurs **formules alternatives** (variantes trigonométriques de Pickover : $\sin(z)+c$, $\cos(z)+c$, $z^p + \sin(z) + c$...)
6. Une **galerie** de biomorphes célèbres
7. Un rendu **Plotly** pour un zoom/pan fluide
8. Une **marche aléatoire** dans l'espace des paramètres $c$
9. Une **animation** de morphose entre deux constantes $c$
10. Un système de **favoris** (JSON) et un export **haute résolution**

> 💡 Astuce : dans JupyterLab, `Kernel > Restart & Run All`, puis direction la section 3 pour bidouiller.


## 0. Imports & configuration

Comme pour le notebook sur l'attracteur de Clifford : détection automatique de **Numba**
(accélération JIT du calcul pixel par pixel, indispensable pour le mode "temps d'échappement" en
haute résolution), **ipywidgets** (sliders interactifs) et **plotly** (zoom/pan fluide).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os, time

plt.rcParams['figure.facecolor'] = '#0d1117'

# --- Numba (accélération JIT) ---------------------------------------------
try:
    from numba import njit
    import numba
    NUMBA_OK = True
except ImportError:
    NUMBA_OK = False
    def njit(*args, **kwargs):
        def wrap(f):
            return f
        if len(args) == 1 and callable(args[0]):
            return args[0]
        return wrap

# --- ipywidgets --------------------------------------------------------
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False

# --- plotly --------------------------------------------------------------
try:
    import plotly.graph_objects as go
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False

print(f"Numba disponible      : {NUMBA_OK}")
print(f"ipywidgets disponible : {WIDGETS_OK}")
print(f"plotly disponible     : {PLOTLY_OK}")
if not NUMBA_OK:
    print(">> conseil : pip install numba  (accélère fortement le mode temps d'échappement)")
if not WIDGETS_OK:
    print(">> conseil : pip install ipywidgets  (nécessaire pour les sliders interactifs)")
if not PLOTLY_OK:
    print(">> conseil : pip install plotly  (nécessaire pour le rendu zoom/pan fluide)")


## 1. Génération — formules disponibles

Quatre variantes de la récurrence sont proposées (toutes des classiques explorées par Pickover) :

| Clé | Formule | Effet visuel |
|---|---|---|
| `power` | $z^{p} + c$ | biomorphe "original" (insectes/étoiles selon $p$) |
| `power_sin` | $z^{p} + \sin(z) + c$ | pattes plus fines et nombreuses |
| `sin` | $\sin(z) + c$ | structures en nappes/ailes |
| `cos` | $\cos(z) + c$ | motifs plus anguleux, symétrie différente |

Chaque formule est compilée séparément en Numba (les fonctions `cmath` ne sont pas dispatchables
dynamiquement à l'intérieur d'une boucle JIT), avec un repli NumPy pur si Numba est absent.
Le calcul renvoie une grille du **temps d'échappement** (nombre d'itérations avant de sortir du
bailout, ou `iters` si le point n'a jamais divergé), ce qui permet à la fois :

- le rendu **binaire** original (`échappé ?  0 : 1`)
- le rendu **coloré par temps d'échappement**, beaucoup plus riche


In [ ]:
import cmath

def _make_njit_kernel(formula):
    """Fabrique un noyau Numba spécialisé pour une formule donnée (évite le dispatch dynamique)."""

    if formula == 'power':
        @njit(fastmath=True)
        def kernel(X, Y, cre, cim, power, bailout, iters):
            h, w = X.shape
            out = np.empty((h, w), dtype=np.int32)
            for j in range(h):
                for i in range(w):
                    zr, zi = X[j, i], Y[j, i]
                    esc = iters
                    for k in range(iters):
                        z = complex(zr, zi) ** power + complex(cre, cim)
                        zr, zi = z.real, z.imag
                        if abs(zr) > bailout or abs(zi) > bailout:
                            esc = k
                            break
                    out[j, i] = esc
            return out
        return kernel

    if formula == 'power_sin':
        @njit(fastmath=True)
        def kernel(X, Y, cre, cim, power, bailout, iters):
            h, w = X.shape
            out = np.empty((h, w), dtype=np.int32)
            for j in range(h):
                for i in range(w):
                    zr, zi = X[j, i], Y[j, i]
                    esc = iters
                    for k in range(iters):
                        zc = complex(zr, zi)
                        z = zc ** power + cmath.sin(zc) + complex(cre, cim)
                        zr, zi = z.real, z.imag
                        if abs(zr) > bailout or abs(zi) > bailout:
                            esc = k
                            break
                    out[j, i] = esc
            return out
        return kernel

    if formula == 'sin':
        @njit(fastmath=True)
        def kernel(X, Y, cre, cim, power, bailout, iters):
            h, w = X.shape
            out = np.empty((h, w), dtype=np.int32)
            for j in range(h):
                for i in range(w):
                    zr, zi = X[j, i], Y[j, i]
                    esc = iters
                    for k in range(iters):
                        z = cmath.sin(complex(zr, zi)) * power + complex(cre, cim)
                        zr, zi = z.real, z.imag
                        if abs(zr) > bailout or abs(zi) > bailout:
                            esc = k
                            break
                    out[j, i] = esc
            return out
        return kernel

    if formula == 'cos':
        @njit(fastmath=True)
        def kernel(X, Y, cre, cim, power, bailout, iters):
            h, w = X.shape
            out = np.empty((h, w), dtype=np.int32)
            for j in range(h):
                for i in range(w):
                    zr, zi = X[j, i], Y[j, i]
                    esc = iters
                    for k in range(iters):
                        z = cmath.cos(complex(zr, zi)) * power + complex(cre, cim)
                        zr, zi = z.real, z.imag
                        if abs(zr) > bailout or abs(zi) > bailout:
                            esc = k
                            break
                    out[j, i] = esc
            return out
        return kernel

    raise ValueError(f"formule inconnue : {formula}")


def _biomorph_numpy(X, Y, cre, cim, power, bailout, iters, formula):
    """Repli NumPy pur (vectorisé), utilisé si Numba est indisponible."""
    Z = X + 1j * Y
    c = complex(cre, cim)
    esc = np.full(Z.shape, iters, dtype=np.int32)
    active = np.ones(Z.shape, dtype=bool)
    for k in range(iters):
        if formula == 'power':
            Z = np.where(active, Z**power + c, Z)
        elif formula == 'power_sin':
            Z = np.where(active, Z**power + np.sin(Z) + c, Z)
        elif formula == 'sin':
            Z = np.where(active, np.sin(Z) * power + c, Z)
        elif formula == 'cos':
            Z = np.where(active, np.cos(Z) * power + c, Z)
        newly = active & ((np.abs(Z.real) > bailout) | (np.abs(Z.imag) > bailout))
        esc[newly] = k
        active &= ~newly
    return esc


_KERNEL_CACHE = {}

def biomorph_escape(xmin, xmax, ymin, ymax, w=450, h=450, power=5, c=complex(0.55, 0.36),
                     bailout=10.0, iters=12, formula='power'):
    """Calcule la grille de temps d'échappement pour les biomorphes de Pickover.
    Retourne (escape_grid, X, Y) où escape_grid contient, pour chaque pixel, le nombre
    d'itérations avant divergence (ou `iters` si le point n'a jamais divergé -> "corps" du biomorphe).
    """
    xs = np.linspace(xmin, xmax, w)
    ys = np.linspace(ymin, ymax, h)
    X, Y = np.meshgrid(xs, ys)

    if NUMBA_OK:
        if formula not in _KERNEL_CACHE:
            _KERNEL_CACHE[formula] = _make_njit_kernel(formula)
        kernel = _KERNEL_CACHE[formula]
        esc = kernel(X, Y, float(c.real), float(c.imag), float(power), float(bailout), int(iters))
    else:
        esc = _biomorph_numpy(X, Y, c.real, c.imag, power, bailout, iters, formula)

    return esc, X, Y


# --- petit benchmark indicatif ---
t0 = time.time()
esc, X, Y = biomorph_escape(-1.2, 1.2, -1.2, 1.2, w=450, h=450, power=5, c=complex(0.55, 0.36))
t1 = time.time()
print(f"Grille 450x450, 12 itérations max, calculée en {t1 - t0:.3f} s "
      f"({'Numba JIT' if NUMBA_OK else 'NumPy pur'})")


## 2. Rendu statique — silhouette binaire & temps d'échappement

- **`mode='binary'`** : reproduit fidèlement le rendu original de Pickover — un pixel appartient au
  biomorphe (blanc/vert) s'il n'a *jamais* divergé après `iters` itérations, noir sinon. C'est ce
  mode qui fait apparaître la silhouette "insecte" caractéristique.
- **`mode='escape'`** : colore chaque pixel selon son temps d'échappement (façon fractale de type
  Mandelbrot), ce qui révèle la structure fine autour du biomorphe — halos, filaments, dégradés.


In [ ]:
def plot_biomorph(xmin, xmax, ymin, ymax, w=450, h=450, power=5, c=complex(0.55, 0.36),
                   bailout=10.0, iters=12, formula='power', mode='binary', cmap='Greens_r',
                   bg_color='#0d1117', ax=None, title=True, figsize=(7, 7)):
    """Affiche un biomorphe de Pickover pour un jeu de paramètres donné."""
    esc, X, Y = biomorph_escape(xmin, xmax, ymin, ymax, w=w, h=h, power=power, c=c,
                                 bailout=bailout, iters=iters, formula=formula)

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=figsize, facecolor=bg_color)
    ax.clear()
    ax.set_facecolor(bg_color)

    if mode == 'binary':
        img = np.where(esc >= iters, 1.0, 0.0)
        ax.imshow(img, cmap=cmap, extent=[xmin, xmax, ymin, ymax], origin='lower')
    else:  # escape
        ax.imshow(esc, cmap=cmap, extent=[xmin, xmax, ymin, ymax], origin='lower')

    ax.set_axis_off()
    ax.set_aspect('equal')
    if title:
        ax.set_title(f"p={power}  c={c.real:.3f}{c.imag:+.3f}i   formule={formula}",
                      color='white', fontsize=10, family='monospace')
    if created_fig:
        plt.tight_layout()
        plt.show()
    return ax


# Rendu du biomorphe d'origine (mode binaire, fidèle au script fourni)
plot_biomorph(-1.2, 1.2, -1.2, 1.2, power=5, c=complex(0.55, 0.36), iters=12,
              mode='binary', cmap='Greens_r')


In [ ]:
# Le même biomorphe, en mode "temps d'échappement" (bien plus détaillé)
plot_biomorph(-1.2, 1.2, -1.2, 1.2, power=5, c=complex(0.55, 0.36), iters=40,
              mode='escape', cmap='viridis')


## 3. 🎛️ Exploration interactive — le "bidouillage" en direct

Tous les paramètres du biomorphe sont exposés via des sliders `ipywidgets` :

| Widget | Rôle |
|---|---|
| `power` | exposant $p$ de la récurrence (contrôle le nombre de "pattes"/symétries) |
| `c_re, c_im` | partie réelle / imaginaire de la constante $c$ |
| `bailout` | seuil de divergence (petites valeurs = silhouettes plus fines) |
| `iters` | nombre d'itérations maximum (plus de détails, mais plus lent) |
| `formula` | `power` / `power_sin` / `sin` / `cos` |
| `mode` | binaire (silhouette) / temps d'échappement (coloré) |
| `zoom` | rayon de la fenêtre d'affichage autour de l'origine (0,0) |
| `resolution` | taille de la grille (rapide ↔ détaillé) |
| `colormap` | palette matplotlib |
| Bouton **🎲 Aléatoire** | tire un nouveau $c$ au hasard |
| Bouton **♻️ Reset** | revient au biomorphe d'origine du script fourni |
| Bouton **💾 Sauver ce favori** | enregistre les paramètres courants dans un fichier JSON |


In [ ]:
if WIDGETS_OK:
    style = {'description_width': '90px'}
    layout_slider = widgets.Layout(width='420px')

    power_s = widgets.FloatSlider(value=5, min=1, max=9, step=1, description='power', style=style, layout=layout_slider, continuous_update=False)
    cre_s = widgets.FloatSlider(value=0.55, min=-2, max=2, step=0.01, description='c_re', style=style, layout=layout_slider, continuous_update=False)
    cim_s = widgets.FloatSlider(value=0.36, min=-2, max=2, step=0.01, description='c_im', style=style, layout=layout_slider, continuous_update=False)
    bailout_s = widgets.FloatSlider(value=10.0, min=1.0, max=30.0, step=0.5, description='bailout', style=style, layout=layout_slider, continuous_update=False)
    iters_s = widgets.IntSlider(value=12, min=3, max=60, step=1, description='iters', style=style, layout=layout_slider, continuous_update=False)

    formula_s = widgets.Dropdown(options=['power', 'power_sin', 'sin', 'cos'], value='power', description='formula', style=style)
    mode_s = widgets.ToggleButtons(options=['binary', 'escape'], value='binary', description='mode', style=style)
    cmap_s = widgets.Dropdown(
        options=['Greens_r', 'viridis', 'plasma', 'inferno', 'magma', 'cividis', 'twilight',
                 'turbo', 'cubehelix', 'gist_heat', 'bone'],
        value='Greens_r', description='colormap', style=style)
    bg_s = widgets.ColorPicker(value='#0d1117', description='bg_color', style=style)

    zoom_s = widgets.FloatSlider(value=1.2, min=0.05, max=3.0, step=0.05, description='zoom', style=style, layout=layout_slider, continuous_update=False)
    res_s = widgets.SelectionSlider(options=[150, 250, 350, 450, 600, 800], value=450, description='resolution', style=style, layout=layout_slider)

    random_btn = widgets.Button(description='🎲 Aléatoire', button_style='warning')
    reset_btn = widgets.Button(description='♻️ Reset', button_style='info')
    save_btn = widgets.Button(description='💾 Sauver ce favori', button_style='success')
    out = widgets.Output()

    DEFAULTS = dict(power=5, c_re=0.55, c_im=0.36, bailout=10.0, iters=12, zoom=1.2, formula='power')

    def redraw(*_):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 7), facecolor=bg_s.value)
            z = zoom_s.value
            plot_biomorph(-z, z, -z, z, w=res_s.value, h=res_s.value,
                          power=power_s.value, c=complex(cre_s.value, cim_s.value),
                          bailout=bailout_s.value, iters=iters_s.value,
                          formula=formula_s.value, mode=mode_s.value,
                          cmap=cmap_s.value, bg_color=bg_s.value, ax=ax)
            plt.show()

    def on_random(_):
        cre_s.value = round(np.random.uniform(-1.5, 1.5), 3)
        cim_s.value = round(np.random.uniform(-1.5, 1.5), 3)
        power_s.value = np.random.choice([2, 3, 4, 5, 6, 7])
        redraw()

    def on_reset(_):
        power_s.value = DEFAULTS['power']
        cre_s.value, cim_s.value = DEFAULTS['c_re'], DEFAULTS['c_im']
        bailout_s.value, iters_s.value = DEFAULTS['bailout'], DEFAULTS['iters']
        zoom_s.value = DEFAULTS['zoom']
        formula_s.value = DEFAULTS['formula']
        redraw()

    FAVORITES_PATH = 'biomorph_favorites.json'

    def on_save(_):
        favs = []
        if os.path.exists(FAVORITES_PATH):
            with open(FAVORITES_PATH) as f:
                favs = json.load(f)
        favs.append(dict(power=power_s.value, c_re=cre_s.value, c_im=cim_s.value,
                          bailout=bailout_s.value, iters=iters_s.value, zoom=zoom_s.value,
                          formula=formula_s.value, mode=mode_s.value, cmap=cmap_s.value,
                          bg=bg_s.value, ts=time.time()))
        with open(FAVORITES_PATH, 'w') as f:
            json.dump(favs, f, indent=2)
        with out:
            print(f"✅ Favori sauvegardé ({len(favs)} au total) dans {FAVORITES_PATH}")

    random_btn.on_click(on_random)
    reset_btn.on_click(on_reset)
    save_btn.on_click(on_save)

    for w_ in [power_s, cre_s, cim_s, bailout_s, iters_s, formula_s, mode_s, cmap_s, bg_s, zoom_s, res_s]:
        w_.observe(redraw, names='value')

    controls = widgets.VBox([
        widgets.HTML("<b>Récurrence</b>"),
        widgets.HBox([power_s, formula_s]),
        widgets.HBox([cre_s, cim_s]),
        widgets.HBox([bailout_s, iters_s]),
        widgets.HTML("<b>Rendu</b>"),
        mode_s,
        widgets.HBox([cmap_s, bg_s]),
        widgets.HBox([zoom_s, res_s]),
        widgets.HBox([random_btn, reset_btn, save_btn]),
    ])

    display(widgets.HBox([controls, out]))
    redraw()
else:
    print("ipywidgets n'est pas installé : section interactive indisponible. "
          "Exécutez `pip install ipywidgets` puis redémarrez le kernel.")


## 4. 🖼️ Galerie de biomorphes célèbres

Quelques jeux de paramètres $(p, c)$ réputés pour leurs silhouettes remarquables — insectes,
étoiles de mer, microbes, motifs végétaux.


In [ ]:
GALLERY = [
    dict(name="Original Pickover", power=5, c=complex(0.55, 0.36), formula='power'),
    dict(name="Étoile à 3 branches", power=3, c=complex(0.4, 0.6), formula='power'),
    dict(name="Insecte classique", power=5, c=complex(-0.2, 0.7), formula='power'),
    dict(name="Microbe filamenteux", power=6, c=complex(0.4, 0.4), formula='power_sin'),
    dict(name="Corail", power=4, c=complex(-0.5, 0.5), formula='power_sin'),
    dict(name="Nappe sinusoïdale", power=2, c=complex(0.3, 0.5), formula='sin'),
    dict(name="Motif angulaire", power=3, c=complex(-0.4, 0.3), formula='cos'),
    dict(name="Araignée", power=7, c=complex(0.3, 0.3), formula='power'),
    dict(name="Fleur", power=5, c=complex(-0.6, 0.2), formula='power_sin'),
]

n_cols = 3
n_rows = int(np.ceil(len(GALLERY) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), facecolor='#0d1117')
axes = np.array(axes).reshape(-1)

for ax, cfg in zip(axes, GALLERY):
    plot_biomorph(-1.3, 1.3, -1.3, 1.3, w=250, h=250, power=cfg['power'], c=cfg['c'],
                  iters=20, formula=cfg['formula'], mode='binary', cmap='Greens_r',
                  ax=ax, title=False)
    ax.set_title(f"{cfg['name']}\np={cfg['power']}  c={cfg['c'].real:.2f}{cfg['c'].imag:+.2f}i",
                  color='white', fontsize=9)

for ax in axes[len(GALLERY):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## 5. 🔍 Rendu Plotly — zoom & pan natifs, très fluides

Le mode "temps d'échappement" se prête particulièrement bien à un rendu `Heatmap` Plotly, qui
permet de zoomer/naviguer dans la structure fine de l'image sans avoir à relancer un calcul (le
zoom Plotly agit sur l'image déjà rendue). Pour un zoom qui *recalcule* réellement la grille à plus
haute résolution sur la zone choisie, utilisez plutôt les sliders `zoom` / `resolution` de la
section 3, ou la fonction `plot_biomorph` directement avec de nouvelles bornes `xmin, xmax, ymin, ymax`.


In [ ]:
def plot_biomorph_plotly(xmin, xmax, ymin, ymax, w=500, h=500, power=5, c=complex(0.55, 0.36),
                          bailout=10.0, iters=30, formula='power', cmap='Viridis', bg_color='#0d1117'):
    if not PLOTLY_OK:
        print("plotly n'est pas installé : `pip install plotly`")
        return None
    esc, X, Y = biomorph_escape(xmin, xmax, ymin, ymax, w=w, h=h, power=power, c=c,
                                 bailout=bailout, iters=iters, formula=formula)

    fig = go.Figure(data=go.Heatmap(
        z=esc, x=np.linspace(xmin, xmax, w), y=np.linspace(ymin, ymax, h),
        colorscale=cmap, showscale=False,
    ))
    fig.update_layout(
        template='plotly_dark',
        paper_bgcolor=bg_color, plot_bgcolor=bg_color,
        width=750, height=750,
        title=f"Biomorphe de Pickover — p={power}, c={c.real:.3f}{c.imag:+.3f}i, formule={formula}",
        xaxis=dict(visible=False, scaleanchor='y'),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=50, b=10),
    )
    fig.show()
    return fig

plot_biomorph_plotly(-1.2, 1.2, -1.2, 1.2, power=5, c=complex(0.55, 0.36), iters=30, cmap='Viridis')


## 6. 🚶 Marche aléatoire dans l'espace des paramètres $c$

Comme pour l'attracteur de Clifford, une marche aléatoire locale dans le plan complexe $c$ permet de
dériver continûment d'un biomorphe vers un voisin proche, en restant dans des zones visuellement
riches (plutôt qu'un tirage uniforme complet, qui donne souvent des formes triviales : disque plein
ou grille vide).


In [ ]:
def random_walk_gallery(start=complex(0.55, 0.36), power=5, formula='power',
                         steps=6, step_size=0.12, seed=7, w=250, res_iters=20):
    rng = np.random.default_rng(seed)
    cs = [start]
    for _ in range(steps - 1):
        delta = complex(rng.normal(scale=step_size), rng.normal(scale=step_size))
        cs.append(cs[-1] + delta)

    n_cols = 3
    n_rows = int(np.ceil(steps / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), facecolor='#0d1117')
    axes = np.array(axes).reshape(-1)
    for ax, cval in zip(axes, cs):
        plot_biomorph(-1.3, 1.3, -1.3, 1.3, w=w, h=w, power=power, c=cval,
                      iters=res_iters, formula=formula, mode='binary', cmap='Greens_r',
                      ax=ax, title=False)
        ax.set_title(f"c={cval.real:.3f}{cval.imag:+.3f}i", color='white', fontsize=8)
    for ax in axes[steps:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return cs

_ = random_walk_gallery()


## 7. 🎞️ Animation — morphose entre deux biomorphes

Interpolation linéaire de la constante $c$ entre un point de départ et un point d'arrivée dans le
plan complexe : le résultat est une **transformation fluide** d'un biomorphe vers un autre, exportée
en GIF (Pillow, aucune dépendance externe type ffmpeg nécessaire).

⚠️ Cellule désactivée par défaut (`RUN_ANIMATION = False`) : le rendu de plusieurs dizaines de
frames en mode temps-d'échappement peut prendre 1 à 2 minutes. Passez la variable à `True` pour
la lancer.


In [ ]:
import matplotlib.animation as animation

RUN_ANIMATION = False  # passez à True pour générer le GIF

def make_morph_gif(c_start, c_end, power=5, formula='power', n_frames=40, w=220,
                    iters=18, out_path='biomorph_morph.gif', cmap='Greens_r',
                    mode='binary', fps=12):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor='#0d1117')

    def update(frame):
        t = frame / (n_frames - 1)
        c = c_start + t * (c_end - c_start)
        plot_biomorph(-1.3, 1.3, -1.3, 1.3, w=w, h=w, power=power, c=c, iters=iters,
                      formula=formula, mode=mode, cmap=cmap, ax=ax, title=False)
        return []

    anim = animation.FuncAnimation(fig, update, frames=n_frames, blit=False)
    anim.save(out_path, writer='pillow', fps=fps)
    plt.close(fig)
    print(f"✅ Animation sauvegardée : {out_path}")
    return out_path

if RUN_ANIMATION:
    make_morph_gif(c_start=complex(0.55, 0.36), c_end=complex(-0.4, 0.6), n_frames=40)
else:
    print("RUN_ANIMATION = False -> cellule ignorée (passez la variable à True pour générer le GIF).")


## 8. 💾 Favoris — sauvegarde & rechargement

Le bouton **💾 Sauver ce favori** de la section interactive écrit les paramètres courants dans
`biomorph_favorites.json`. Les cellules ci-dessous listent ces favoris et permettent d'en recharger
n'importe lequel directement dans un rendu.


In [ ]:
FAVORITES_PATH = 'biomorph_favorites.json'

def list_favorites():
    if not os.path.exists(FAVORITES_PATH):
        print("Aucun favori enregistré pour le moment (utilisez le bouton 💾 dans la section 3).")
        return []
    with open(FAVORITES_PATH) as f:
        favs = json.load(f)
    for i, fdict in enumerate(favs):
        print(f"[{i}] power={fdict['power']} c={fdict['c_re']:.3f}{fdict['c_im']:+.3f}i "
              f"formula={fdict.get('formula', '-')} mode={fdict.get('mode', '-')}")
    return favs

def render_favorite(index, w=450):
    favs = list_favorites()
    if not favs or index >= len(favs):
        print("Index invalide.")
        return
    f = favs[index]
    z = f.get('zoom', 1.2)
    plot_biomorph(-z, z, -z, z, w=w, h=w, power=f['power'],
                  c=complex(f['c_re'], f['c_im']), bailout=f.get('bailout', 10.0),
                  iters=f.get('iters', 12), formula=f.get('formula', 'power'),
                  mode=f.get('mode', 'binary'), cmap=f.get('cmap', 'Greens_r'),
                  bg_color=f.get('bg', '#0d1117'))

favs = list_favorites()
# Exemple : render_favorite(0)


## 9. 🖨️ Export haute résolution

Fonction utilitaire pour exporter un biomorphe en très haute résolution (impression, fond d'écran,
usage artistique).


In [ ]:
def export_hd(power, c, filename='biomorph_hd.png', w=1600, h=1600, iters=40,
              formula='power', mode='escape', cmap='viridis', dpi=200,
              figsize=(10, 10), bg_color='#0d1117', zoom=1.2):
    fig, ax = plt.subplots(figsize=figsize, facecolor=bg_color, dpi=dpi)
    plot_biomorph(-zoom, zoom, -zoom, zoom, w=w, h=h, power=power, c=c, iters=iters,
                  formula=formula, mode=mode, cmap=cmap, bg_color=bg_color, ax=ax, title=False)
    plt.savefig(filename, dpi=dpi, facecolor=bg_color, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    print(f"✅ Export haute résolution enregistré : {filename}")

# Exemple (décommentez pour lancer un export ~quelques dizaines de secondes) :
# export_hd(5, complex(0.55, 0.36), filename='biomorph_hd.png', w=1600, h=1600)


## 10. 🎁 Bonus — comparaison des 4 formules sur le même $c$

Pour bien visualiser l'impact du choix de formule, voici le **même** couple $(p, c)$ décliné sur les
quatre variantes disponibles.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), facecolor='#0d1117')
c_demo = complex(0.4, 0.4)
for ax, f in zip(axes, ['power', 'power_sin', 'sin', 'cos']):
    plot_biomorph(-1.3, 1.3, -1.3, 1.3, w=280, h=280, power=5, c=c_demo,
                  iters=22, formula=f, mode='binary', cmap='Greens_r', ax=ax, title=False)
    ax.set_title(f, color='white', fontsize=11)
plt.tight_layout()
plt.show()


## 🔚 Conclusion & pistes pour aller plus loin

- Le paramètre `power` contrôle grossièrement le nombre de "pattes" du biomorphe : essayez de
  grandes valeurs impaires (7, 9) pour des créatures très filamenteuses.
- En mode `escape`, baissez `bailout` (2 à 5) pour révéler des halos concentriques fins autour du corps.
- Le fichier `biomorph_favorites.json` généré par ce notebook peut servir de base à une **galerie
  permanente** de vos plus belles trouvailles — comme pour l'attracteur de Clifford.
- Pour aller plus loin : biomorphes en couleur RVB (un canal par formule ou par nombre d'itérations
  jusqu'à divergence sur chaque axe séparément), ou biomorphes "hybrides" alternant deux formules à
  chaque itération.

Bonne exploration entomologique ! 🐞
